# 01 - Configuración del Pipeline: esquemas Silver, DQL, Gold y Auditoría

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

Primera tarea del pipeline. Crea, dentro del catálogo `datos_empresas` que ya existe desde el TP1, los esquemas que usan los demás notebooks:

| Esquema | Propósito |
|---|---|
| `bronze` | **Ya existe (TP1).** Datos crudos de las APIs. Es la entrada del pipeline y no se modifica. |
| `silver` | Datos limpios, tipados y validados. Solo llegan los registros que cumplen todas las reglas de calidad. |
| `dql` | *Data Quality Layer*: registros rechazados, con la fila original de Bronze, el motivo de rechazo y la fecha de detección. |
| `gold` | Modelo analítico (dimensión, hechos y agregados) construido únicamente desde Silver. |
| `auditoria` | Bitácora de cada ejecución: filas insertadas / actualizadas / eliminadas por tabla. Es la evidencia de idempotencia. |

Todas las sentencias usan `IF NOT EXISTS`, así que esta tarea también es idempotente.

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Verificar que existen las tablas Bronze (entrada del pipeline)

In [ ]:
%python
faltantes = [t for t in (BRONZE_CIIU, BRONZE_RUES, BRONZE_SECOP, BRONZE_TRM) if not spark.catalog.tableExists(t)]
if faltantes:
    raise RuntimeError(f"Faltan tablas Bronze; ejecute primero la ingesta del TP1: {faltantes}")

for t in (BRONZE_CIIU, BRONZE_RUES, BRONZE_SECOP, BRONZE_TRM):
    print(f"{t}: {spark.table(t).count():,} filas")

---
## Paso 2: Crear los esquemas

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS datos_empresas.silver
COMMENT 'Capa Silver - Datos limpios, tipados y validados de RUES, SECOP, CIIU y TRM. Las filas que incumplen alguna regla de calidad no llegan aqui: van al esquema dql.';

CREATE SCHEMA IF NOT EXISTS datos_empresas.dql
COMMENT 'Data Quality Layer - Registros de Bronze rechazados durante el procesamiento hacia Silver, con la fila original, el motivo (_motivo_rechazo) y la fecha de deteccion (_fecha_rechazo).';

CREATE SCHEMA IF NOT EXISTS datos_empresas.gold
COMMENT 'Capa Gold - Modelo analitico (dimension de empresas, hechos de contratos y agregados mensuales) construido a partir de Silver.';

CREATE SCHEMA IF NOT EXISTS datos_empresas.auditoria
COMMENT 'Bitacora del pipeline: metricas de cada escritura por ejecucion. Evidencia de idempotencia.';

---
## Paso 3: Crear la tabla de auditoría

Una fila por `(id_ejecucion, tabla)`. Las columnas de escritura las llena `escribir_merge` en cada notebook; las de validación las llena `09_Auditoria_Ejecucion` al final del pipeline.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS datos_empresas.auditoria.de_pipeline_ejecuciones (
  id_ejecucion        STRING    COMMENT 'Id de la corrida (job.run_id o manual)',
  capa                STRING    COMMENT 'bronze | silver | dql | gold',
  tabla               STRING    COMMENT 'Nombre completo de la tabla',
  version_delta       BIGINT    COMMENT 'Version Delta despues de la escritura',
  operacion           STRING    COMMENT 'Operacion Delta registrada (MERGE)',
  filas_origen        BIGINT    COMMENT 'Filas del DataFrame de origen del MERGE',
  filas_insertadas    BIGINT    COMMENT 'numTargetRowsInserted',
  filas_actualizadas  BIGINT    COMMENT 'numTargetRowsUpdated',
  filas_eliminadas    BIGINT    COMMENT 'numTargetRowsDeleted',
  fecha_escritura     TIMESTAMP COMMENT 'Momento de la escritura',
  filas_tabla         BIGINT    COMMENT 'Filas de la tabla al final de la ejecucion',
  llaves_distintas    BIGINT    COMMENT 'Llaves de negocio distintas (debe ser igual a filas_tabla)',
  filas_duplicadas    BIGINT    COMMENT 'filas_tabla - llaves_distintas (debe ser 0)',
  checksum            DECIMAL(38,0) COMMENT 'Suma de xxhash64(_hash_registro): huella del contenido de la tabla',
  validacion_ok       BOOLEAN   COMMENT 'Resultado de las validaciones de 09_Auditoria_Ejecucion',
  fecha_validacion    TIMESTAMP COMMENT 'Momento de la validacion'
)
COMMENT 'Bitacora del pipeline datos_empresas: una fila por (id_ejecucion, tabla).';

---
## Paso 4 (opcional): Reiniciar las tablas del pipeline

Solo para empezar desde cero en una demostración. **No borra Bronze ni la auditoría.** Por defecto (`reiniciar_tablas = no`) no hace nada.

In [ ]:
%python
dbutils.widgets.dropdown("reiniciar_tablas", "no", ["no", "si"], "Reiniciar Silver/DQL/Gold")

if dbutils.widgets.get("reiniciar_tablas") == "si":
    tablas = ([f["silver"] for f in FUENTES.values()] + [f["dql"] for f in FUENTES.values()] + list(TABLAS_GOLD))
    for t in tablas:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        print(f"Eliminada: {t}")
else:
    print("reiniciar_tablas = no -> no se elimina ninguna tabla.")

In [ ]:
%sql
SHOW SCHEMAS IN datos_empresas;

In [ ]:
%python
finalizar({"notebook": "01_Configuracion_Pipeline", "estado": "OK"})